# Textbook canonical — AIMA umbrella world

Russell & Norvig's *Artificial Intelligence : A Modern Approach* uses a 2-state HMM throughout Chapter 14 to illustrate filtering, smoothing, and prediction. The world : it either rains or it doesn't ; the only observation is whether the security guard brings an umbrella to work.

This notebook reproduces the canonical values from AIMA Sections 14.2.2 (filtering) and 14.2.4 (smoothing). Our `predict_proba` must match the published numbers — that's how we know the forward-backward implementation is correct.

**No fitting** — we pin model parameters at the textbook values and only run inference.

## 1. Build the umbrella HMM

From AIMA Section 14.2 :

| Parameter | Value |
|---|---|
| `P(rain_t | rain_{t-1})` | 0.7 |
| `P(sun_t  | sun_{t-1})`  | 0.7 |
| `P(umbrella | rain)` | 0.9 |
| `P(umbrella | sun)`  | 0.2 |
| `P(rain_0)` | 0.5 |

In [ ]:
import numpy as np
from hmm_core.fit.multinomial import ConstrainedMultinomialHMM

model = ConstrainedMultinomialHMM(
    n_components=2, n_features=2, random_state=0, transmat_mask=None,
)
model.startprob_ = np.array([0.5, 0.5])
model.transmat_ = np.array([
    [0.7, 0.3],   # from rain
    [0.3, 0.7],   # from sun
])
model.emissionprob_ = np.array([
    [0.1, 0.9],   # rain : 10% no-umbrella, 90% umbrella
    [0.8, 0.2],   # sun  : 80% no-umbrella, 20% umbrella
])
print("States       : 0=rain, 1=sun")
print("Observations : 0=no_umbrella, 1=umbrella")

## 2. Canonical 5-step observation sequence

AIMA tracks the guard over 5 consecutive days. On day 3 he doesn't bring an umbrella ; every other day he does.

In [ ]:
observations = np.array([[1], [1], [0], [1], [1]])
print("Day :   1   2   3   4   5")
print("Obs :  " + "   ".join(("U" if o[0] == 1 else "-") for o in observations))

## 3. Smoothing — `P(rain_t | x_{1:5})`

Forward-backward gives smoothed posteriors at every timestep, using **all** observations. AIMA Figure 14.5 gives the canonical values that our implementation must reproduce within rounding tolerance.

In [ ]:
posteriors = model.predict_proba(observations)
p_rain = posteriors[:, 0]

expected = np.array([0.867, 0.820, 0.306, 0.821, 0.867])

print("Day | P(rain) ours | P(rain) AIMA | diff")
print("----+--------------+--------------+--------")
for t, (ours, ref) in enumerate(zip(p_rain, expected), start=1):
    print(f"  {t} |    {ours:6.3f}    |    {ref:5.3f}     | {ours - ref:+.4f}")

np.testing.assert_allclose(p_rain, expected, atol=5e-3)
print("\nOK — matches AIMA Section 14.2.4 smoothing values.")

## 4. Filtering — `P(rain_t | x_{1:t})`

Filtering uses only the **past** up to time `t`. AIMA Section 14.2.2 gives different (and slightly lower) values than smoothing : you don't get to use the future to refine your belief.

We obtain filtered values by running `predict_proba` on growing prefixes and reading off the last entry each time.

In [ ]:
expected_filtered = np.array([0.818, 0.883, 0.190, 0.731, 0.867])
filtered = np.empty(5)
for t in range(1, 6):
    posteriors_prefix = model.predict_proba(observations[:t])
    filtered[t - 1] = posteriors_prefix[-1, 0]

print("Day | P(rain | past) ours | AIMA  | diff")
print("----+---------------------+-------+--------")
for t, (ours, ref) in enumerate(zip(filtered, expected_filtered), start=1):
    print(f"  {t} |       {ours:6.3f}        | {ref:5.3f} | {ours - ref:+.4f}")

np.testing.assert_allclose(filtered, expected_filtered, atol=5e-3)
print("\nOK — matches AIMA Section 14.2.2 filtering values.")

## 5. The day-3 effect

On day 3 the guard brings no umbrella, so our belief in *rain* collapses : filtered drops from 0.883 to 0.190. Smoothing for day 2 (knowing day 3 was dry-looking) also drops slightly versus the filtered estimate (0.820 vs 0.883).

This is the *signature behavior* of forward-backward : future evidence influences past beliefs.

## Next

- **Durbin dishonest casino** (`08_textbook_dishonest_casino.ipynb`) — Viterbi accuracy on a harder, longer sequence.
- **Full validation suite** : `validation/test_v3_textbook_canonical.py` covers AIMA umbrella, Durbin casino, and the Eisner ice cream HMM.